In [2]:
import pandas as pd

df = pd.read_csv('full_data.csv')

jan6_df = pd.read_csv('jan6_articles.csv')
jan6_df.head(1)

,row_id,text-topic,text-topic-exp,text-entity-name,text-entity-gender,text-entity-sentiment,text-entity-sentiment-exp,text-generic-frame,text-generic-frame-exp,text-issue-frame,...,excerpt_length,gpt_title_frame,gpt_excerpt_primary_frame,gpt_excerpt_secondary_frame,gpt_demands_present,gpt_protester_quote,gpt_authority_quote,gpt_thematic_present,frames_error,binary_error
0,10,Politics,"The article discusses political matters, such ...",Chris Christie,male,negative,Christie criticizes other Republican candidate...,"['Morality', 'Political', 'Crime and punishment']",The article discusses the moral responsibility...,Political Loyalty vs Accountability,...,1380,residual,protester_violence,property_damage,0,0,1,0,NaN,NaN


In [ ]:
jan6_pattern = r"""(?ix)
\b(
    jan(?:uary)?\.?\s*6(?:th)? |
    january\s+sixth |
    j6 |
    capitol\s+(?:riot|riots|attack|assault|siege|insurrection|breach|storming) |
    (?:riot|riots|attack|assault|siege|insurrection|breach|storming)\s+(?:of|on|at)\s+(?:the\s+)?(?:u\.?s\.?\s+)?capitol |
    storm(?:ed|ing)?\s+(?:the\s+)?(?:u\.?s\.?\s+)?capitol
)\b
"""

text = (
    merged_df["title"].fillna("").astype(str) + " " +
    merged_df["protest_excerpt"].fillna("").astype(str)
)

jan6_df = merged_df[
    text.str.contains(jan6_pattern, regex=True, na=False)
].copy()

print(f"January 6 candidate articles: {len(jan6_df):,}")

# Save all columns
jan6_df.to_csv("jan6_articles.csv", index=False, encoding="utf-8")

print(f"Saved {len(jan6_df):,} articles to jan6_articles.csv")

In [ ]:
excerpt_match = df['protest_excerpt'].fillna('').str.contains(
    jan6_pattern, regex=True
)

title_match = df['title'].fillna('').str.contains(
    jan6_pattern, regex=True
)

excerpt_only_df = df[excerpt_match & ~title_match].copy()
both_df = df[excerpt_match & title_match].copy()

print(f"Excerpt-only matches: {len(excerpt_only_df):,}")
print(f"Both Title and Excerpt matches: {len(both_df):,}")

In [3]:
# Political leaning order
lean_order = [
    "left",
    "left_lean",
    "center",
    "right_lean",
    "right"
]

# Your four binary variables — change names if needed
binary_cols = [
    "gpt_demands_present",
    "gpt_protester_quote",
    "gpt_authority_quote",
    "gpt_thematic_present"
]

In [4]:
jan6_excerpt_frames = (
    jan6_df["gpt_excerpt_primary_frame"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .rename("Percent")
    .to_frame()
)

jan6_excerpt_frames["N"] = jan6_df["gpt_excerpt_primary_frame"].value_counts()


jan6_excerpt_frames

,Percent,N
gpt_excerpt_primary_frame,,
protester_violence,62.2,909
residual,11.4,166
disruption,8.1,118
impactful,7.2,105
harm_to_protesters,4.4,65
grievance,2.3,33
property_damage,1.9,28
widespread_support,1.4,21
peaceful,1.1,16


In [5]:
jan6_lean = (
    jan6_df["political_leaning"]
    .value_counts()
    .reindex(lean_order)
    .rename("N")
    .to_frame()
)

jan6_lean["Percent"] = (
    jan6_lean["N"] / jan6_lean["N"].sum() * 100
).round(1)

jan6_lean

,N,Percent
political_leaning,,
left,203,13.9
left_lean,590,40.4
center,280,19.2
right_lean,128,8.8
right,260,17.8


left lean is also the most common in the total df

Excerpt Frame by Political Lean

In [6]:
jan6_frames_by_lean = (
    pd.crosstab(
        jan6_df["gpt_excerpt_primary_frame"],
        jan6_df["political_leaning"],
        normalize="columns"
    )
    .mul(100)
    .round(1)
    .reindex(columns=lean_order)
)

jan6_frames_by_lean

political_leaning,left,left_lean,center,right_lean,right
gpt_excerpt_primary_frame,,,,,
disruption,3.9,6.4,10.4,12.5,10.4
grievance,1.5,1.9,2.9,1.6,3.5
harm_to_protesters,3.4,2.7,2.5,6.2,10.4
impactful,5.9,4.4,8.6,7.0,13.1
peaceful,2.0,0.7,1.1,1.6,1.2
property_damage,0.5,1.9,2.1,0.8,3.5
protester_violence,70.9,72.2,55.4,53.1,44.6
residual,9.9,8.0,15.7,16.4,13.1
widespread_support,2.0,1.9,1.4,0.8,0.4


Interesting-- protester violence frame most used by left and left_lean, while right and right_lean go for harm_to_protesters.

In [7]:
jan6_binary_by_lean = (
    jan6_df
    .groupby("political_leaning")[binary_cols]
    .mean()
    .mul(100)
    .round(1)
    .T
    .reindex(columns=lean_order)
)

jan6_binary_by_lean

political_leaning,left,left_lean,center,right_lean,right
gpt_demands_present,62.6,66.8,63.2,60.2,51.5
gpt_protester_quote,28.1,44.9,32.1,35.2,34.6
gpt_authority_quote,87.2,86.1,85.4,88.3,88.5
gpt_thematic_present,53.2,36.6,42.5,39.8,52.3


Political spectrum seems to not be predictive of quotes (both extremes have similar rates) or thematic (extremes have highest rates), but demands are generally higher for left and left_lean outlets. 

authority quotes higher for jan6 subset than overall